# Lane 4: Investment Research Agent

Owner: Ian Schmitt. Development notebook; finished cells are copied into `notebook.ipynb` section 4 at assembly. See `docs/` and `setup.md`.

In [ ]:
import json
import sys
sys.path.insert(0, "..")  # this notebook lives in dev/; make src/ importable

from src import llm, tools
from src.llm import chat, handoff


Plans its research, calls tools dynamically, reflects on its output, and learns across runs.

**Contract:** `ResearchAgent.plan(symbol) -> [{"tool", "args", "why"}]` (memory injected into the prompt, max 8 steps); `run(symbol) -> dict` (dispatches by name over `tools.TOOLS` plus `run_chain` and `route_and_analyze`, logs every call, writes the report through Lane 3's loop); `reflect(symbol, run_result) -> {"score", "note", "lesson"}` (takes `run()`'s output, the plan, tool log, report, and score history; scores the report with Lane 3's `evaluate` and writes a symbol-specific note plus an optional general lesson); `remember(symbol, note, lesson=None)` (appends the note under the symbol and the lesson to `lessons` in `self.memory`, a dict that starts empty on every notebook run).

**Demonstration:** AAPL twice, NVDA once. Run 2's plan must visibly change because of the stored notes; run the planner at temperature 0 so memory is the only input that differs.

During development, import the other lanes' functions from their script exports, which live next to this notebook (regenerate with `uv run jupyter nbconvert --to script dev/chain.ipynb`):

```python
from chain import run_chain
from route import route_and_analyze
from evaluate import evaluator_optimizer
```

### Stand-ins (development only)

Until the lanes deliver on Oct 4, these four functions stand in for them with the same names, arguments, and return shapes, so the agent can be built and run now. `run_chain` and `route_and_analyze` make no LLM calls; `evaluate` and `evaluator_optimizer` make real ones so reflection has a real score to work with. When the exports exist, replace this cell with the three imports above. This cell is never copied into `notebook.ipynb`.


In [ ]:
# DEV ONLY. Same names, arguments, and return shapes as the lanes' functions.


# Lane 1 stand-in: the cached news, every article labeled neutral, no entities.
def run_chain(symbol):
    articles = [{**a, "sentiment": "neutral", "topic": "news", "entities": []} for a in tools.get_news(symbol)]
    return {"articles": articles, "summary": f"Stand-in summary of {len(articles)} {symbol} headlines."}


# Lane 2 stand-in: every item routed to news, with a placeholder analysis.
def route_and_analyze(symbol, items):
    return [{"item": item, "route": "news", "analysis": f"Stand-in analysis of: {item['title']}"} for item in items]


# Lane 3 stand-ins: one real Critic call, and at most one real revision.
def evaluate(text, criteria):
    system = (
        "You are the Critic. Check the report against each criterion. Reply as JSON: "
        '{"score": 1-10, "pass": true or false, "feedback": ["<criterion>: <what is missing or wrong>", ...]}. '
        "pass is true only if every criterion is met; feedback is empty when pass is true."
    )
    return chat(system, f"CRITERIA:\n{criteria}\n\nREPORT:\n{text}", json_mode=True, agent="Critic")


def evaluator_optimizer(generate, criteria, max_rounds=2):
    text = generate()
    check = evaluate(text, criteria)
    history = [{"round": 0, **check}]
    if not check["pass"]:
        system = ("You are the Writer. Revise the report to address every note and change nothing else. "
                  "Reply with the revised report only, no commentary on what changed.")
        text = chat(system, f"NOTES:\n{json.dumps(check['feedback'])}\n\nREPORT:\n{text}", agent="Writer")
        check = evaluate(text, criteria)
        history.append({"round": 1, **check})
    return {"final": text, "history": history}


In [ ]:
# Every step the planner can choose, by name. The executor looks the name up here and calls it.
# Built once when this cell runs, so the lane functions must be defined (or imported) above it.
STEPS = {**tools.TOOLS, "run_chain": run_chain, "route_and_analyze": route_and_analyze}

# What the Coordinator reads when planning: what each step returns, what it costs, and its args.
# The args shown are the cached ones; any other value fetches live from Yahoo.
MENU = """\
get_prices  args {"period": "6mo"}  Daily closes plus last_close, change_pct, high, low, avg_volume. No LLM calls.
get_fundamentals  args {}  Name, sector, market cap and revenue (billions), P/E, revenue and earnings growth and
    profit margin (percent), 52-week range. No LLM calls.
get_news  args {"limit": 10}  Recent headlines with summaries, unlabeled. No LLM calls.
run_chain  args {}  Labels each recent article's sentiment and topic, extracts entities, and writes one summary.
    One LLM call per article.
route_and_analyze  args {}  Sends each article to an earnings, news, or market specialist for a short analysis.
    Two LLM calls per article. Uses run_chain's labeled articles if run_chain ran earlier in the plan.
"""


In [ ]:
class ResearchAgent:
    # Memory lives on the agent and starts empty, so every notebook run is a cold start.
    # symbols holds notes per symbol; lessons holds notes that apply to any symbol.
    def __init__(self):
        self.memory = {"symbols": {}, "lessons": []}

    # Ask the LLM for an ordered list of steps {"tool", "args", "why"}, with memory in the prompt.
    def plan(self, symbol):
        raise NotImplementedError

    # Execute the plan by tool name, write the report through Lane 3's loop, return everything.
    def run(self, symbol):
        raise NotImplementedError

    # Score run()'s report and write {"score", "note", "lesson"}; the note is specific to
    # this symbol, the lesson (or None) applies to any symbol.
    def reflect(self, symbol, run_result):
        raise NotImplementedError

    # Append the note under this symbol and the lesson to the shared lessons list.
    def remember(self, symbol, note, lesson=None):
        raise NotImplementedError


## Demonstration

Run the function on cached data and produce the table and figure described above. Keep printed output narrow enough for a PDF page.

Put demo code inside the `if __name__ == "__main__":` block below. It runs normally in this notebook, but not when another lane imports this notebook's `.py` export, so importing your functions never re-runs your LLM calls and plots.

In [ ]:
if __name__ == "__main__":
    # demo code goes here, indented under this line
    pass
